# Lab 64 (solution): Nugget-based coverage and citation

Reference implementation. Decompose an information need into nuggets, label support, and score coverage (recall) and citation (sentence-support rate) as two orthogonal axes - the AutoNuggetizer / Auto-ARGUE approach that is the 2026 standard for long-form RAG evaluation. Concept: [concepts/rag/nugget-evaluation.md](../../../concepts/rag/nugget-evaluation.md); math: [math-foundations/19](../../../math-foundations/19-nugget-coverage-metrics.md).

## Step 0: Setup

In [ ]:
from nugget_eval import (evaluate, coverage, sentence_support_rate, support_label, NUGGETS,
    ANSWER_CITE_GOOD_MISS, ANSWER_COVER_CITE_BAD, ANSWER_BALANCED)
# Nugget-based evaluation (AutoNuggetizer / Auto-ARGUE style): decompose the information need into
# atomic nuggets, label how well the answer supports each, and score two orthogonal axes -
# coverage (recall over nuggets) and sentence-support rate (citation precision).
print(f"{len(NUGGETS)} nuggets; vital:", [n['id'] for n in NUGGETS if n['weight']=='vital'])

## Step 1: Cites well, misses a nugget (high citation, low coverage)

In [ ]:
# Answer A cites carefully but misses a vital nugget. High citation, low coverage - the TREC 2025
# pattern: citation accuracy is largely solved, coverage is the hard part.
a = evaluate(*ANSWER_CITE_GOOD_MISS)
print(f"coverage(vital) {a['vital_coverage']:.2f}  SSR {a['sentence_support_rate']:.2f}  missing {a['missing_vital']}")

## Step 2: Covers all, cites badly (high coverage, zero citation)

In [ ]:
# Answer B covers every nugget but cites the wrong document. High coverage, zero citation - a
# different bug entirely (attribution, not retrieval).
b = evaluate(*ANSWER_COVER_CITE_BAD)
print(f"coverage(vital) {b['vital_coverage']:.2f}  SSR {b['sentence_support_rate']:.2f}  unsupported {len(b['unsupported_sentences'])}")

## Step 3: Both - and why one score is not enough

In [ ]:
# Answer C covers and cites. The point: A and B sit at opposite corners of a 2x2, so one quality
# score cannot tell them apart - report coverage and citation separately, and attribute the failure.
c = evaluate(*ANSWER_BALANCED)
print(f"coverage(vital) {c['vital_coverage']:.2f}  SSR {c['sentence_support_rate']:.2f}")
print()
print(f"{'answer':18}{'coverage':>10}{'SSR':>7}")
for name,r in [("cite-good/miss",a),("cover/cite-bad",b),("balanced",c)]:
    print(f"{name:18}{r['vital_coverage']:>10.2f}{r['sentence_support_rate']:>7.2f}")

## Step 4: Support labels (Full / Partial / No)

In [ ]:
# Support is labeled Full / Partial / No for partial credit (the TREC nugget tradition).
for n in NUGGETS:
    print(f"  {n['id']} [{n['weight']:5}] {support_label(n, ANSWER_CITE_GOOD_MISS[0]):8} {n['text']}")

## What you built

The 2026 standard for evaluating long-form and report-generation RAG: nugget-based coverage and citation, after the TREC RAG track's AutoNuggetizer and Auto-ARGUE. You decompose an information need into vital/okay nuggets, label each Full/Partial/No support for partial credit, and score two orthogonal axes - **coverage** (recall over the nuggets a good answer should support) and **sentence-support rate** (the fraction of answer sentences actually supported by their cited evidence). The three answers land at different corners: one cites perfectly but misses a vital nugget (high citation, low coverage), one covers everything but cites the wrong document (high coverage, zero citation), one does both. This is the TREC 2025 finding made concrete - citation is largely solved when systems add checks, while coverage lags - and the two are different bugs: coverage is mostly a retrieval problem (you did not retrieve the diverse evidence the nuggets need), citation is mostly a generation problem (you cited the wrong thing). A single quality score hides which one you have.

**Where this simplifies:** the support labeler and the citation judge are deterministic stand-ins so the lab runs offline; `assign_with_judge` is the guarded seam for an AutoNuggetizer-style LLM assignment (the listwise Full/Partial/No judgment a model like GPT-4.1 makes in the real pipeline). The metric design - weighted nugget recall for coverage, sentence-level support for citation, reported separately with an attribution - is the deliverable and is unchanged with a real judge. Math: [math-foundations/19](../../../math-foundations/19-nugget-coverage-metrics.md); concept: [concepts/rag/nugget-evaluation.md](../../../concepts/rag/nugget-evaluation.md).